# LightGBM с Grid Search для детекции депрессии

## Импорты

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix, make_scorer
import lightgbm as lgb
import xgboost as xgb
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def calculateMetrics(yTrue: np.ndarray, yPred: np.ndarray, modelName: str = "Model") -> float:
    f1Macro = f1_score(yTrue, yPred, average='macro')
    f1Binary = f1_score(yTrue, yPred, pos_label=1)
    print(f"{modelName.upper()} → F1-Macro: {f1Macro:.5f} | F1-Binary: {f1Binary:.5f}")
    return f1Macro

## Загрузка данных

In [ ]:
trainDataFrame = pd.read_csv('data/train.csv')
testDataFrame = pd.read_csv('data/test.csv')

print(f"Train shape: {trainDataFrame.shape}")
print(f"Test shape: {testDataFrame.shape}")
print(f"\nClass distribution:")
print(trainDataFrame['label'].value_counts())
print(f"\nClass balance: {trainDataFrame['label'].value_counts(normalize=True)}")

## Предобработка текста

In [ ]:
def cleanText(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' MENTION ', text)
    text = re.sub(r'#\w+', ' HASHTAG ', text)
    text = re.sub(r'\d+', ' NUMBER ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extractTextFeatures(text: str) -> dict:
    if pd.isna(text):
        text = ""
    text = str(text)
    
    return {
        'textLength': len(text),
        'wordCount': len(text.split()),
        'avgWordLength': np.mean([len(w) for w in text.split()]) if text.split() else 0,
        'upperCaseRatio': sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0,
        'exclamationCount': text.count('!'),
        'questionCount': text.count('?'),
        'dotCount': text.count('.'),
        'commaCount': text.count(','),
        'quoteCount': text.count('"') + text.count("'"),
    }

In [ ]:
trainDataFrame['title'] = trainDataFrame['title'].fillna('')
trainDataFrame['body'] = trainDataFrame['body'].fillna('')
testDataFrame['title'] = testDataFrame['title'].fillna('')
testDataFrame['body'] = testDataFrame['body'].fillna('')

trainDataFrame['cleanedTitle'] = trainDataFrame['title'].apply(cleanText)
trainDataFrame['cleanedBody'] = trainDataFrame['body'].apply(cleanText)
testDataFrame['cleanedTitle'] = testDataFrame['title'].apply(cleanText)
testDataFrame['cleanedBody'] = testDataFrame['body'].apply(cleanText)

trainDataFrame['combinedText'] = trainDataFrame['cleanedTitle'] + ' ' + trainDataFrame['cleanedBody']
testDataFrame['combinedText'] = testDataFrame['cleanedTitle'] + ' ' + testDataFrame['cleanedBody']

print("Извлечение дополнительных признаков...")
trainTextFeatures = trainDataFrame['body'].apply(extractTextFeatures).apply(pd.Series)
testTextFeatures = testDataFrame['body'].apply(extractTextFeatures).apply(pd.Series)

print(f"Дополнительных признаков: {trainTextFeatures.shape[1]}")

## Векторизация

In [ ]:
tfidfVectorizerWords = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    use_idf=True
)

tfidfVectorizerChars = TfidfVectorizer(
    max_features=5000,
    ngram_range=(2, 4),
    analyzer='char',
    sublinear_tf=True
)

print("Векторизация текста (word-level)...")
trainTfidfWords = tfidfVectorizerWords.fit_transform(trainDataFrame['combinedText'])
testTfidfWords = tfidfVectorizerWords.transform(testDataFrame['combinedText'])

print("Векторизация текста (char-level)...")
trainTfidfChars = tfidfVectorizerChars.fit_transform(trainDataFrame['combinedText'])
testTfidfChars = tfidfVectorizerChars.transform(testDataFrame['combinedText'])

print(f"Word TF-IDF shape: {trainTfidfWords.shape}")
print(f"Char TF-IDF shape: {trainTfidfChars.shape}")

In [ ]:
from scipy.sparse import hstack

trainFeaturesSparse = hstack([
    trainTfidfWords,
    trainTfidfChars,
    trainTextFeatures.values
])

testFeaturesSparse = hstack([
    testTfidfWords,
    testTfidfChars,
    testTextFeatures.values
])

yTarget = trainDataFrame['label'].values

print(f"Итоговая размерность признаков: {trainFeaturesSparse.shape}")
print(f"Количество классов: {len(np.unique(yTarget))}")

## Разделение на train/validation

In [ ]:
xTrain, xVal, yTrain, yVal = train_test_split(
    trainFeaturesSparse, yTarget,
    test_size=0.2,
    random_state=42,
    stratify=yTarget
)

print(f"Train size: {xTrain.shape[0]}")
print(f"Validation size: {xVal.shape[0]}")
print(f"Train class distribution: {np.bincount(yTrain.astype(int))}")
print(f"Val class distribution: {np.bincount(yVal.astype(int))}")

## Grid Search для LightGBM

In [ ]:
from itertools import product

paramGrid = {
    'n_estimators': [500, 800, 1000, 1200],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.02, 0.03, 0.05],
}

fixedParams = {
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print("Запуск Grid Search на train/val split...\n")

bestF1Score = 0
bestParams = None
allResults = []

paramCombinations = list(product(
    paramGrid['n_estimators'],
    paramGrid['max_depth'],
    paramGrid['learning_rate']
))

totalConfigs = len(paramCombinations)
print(f"Всего конфигураций для проверки: {totalConfigs}\n")

for configIndex, (nEstimators, maxDepth, learningRate) in enumerate(paramCombinations, 1):
    currentModel = xgb.XGBClassifier(
        n_estimators=nEstimators,
        max_depth=maxDepth,
        learning_rate=learningRate,
        **fixedParams
    )
    
    currentModel.fit(xTrain, yTrain)
    valPred = currentModel.predict(xVal)
    f1Val = f1_score(yVal, valPred, average='macro')
    
    currentParams = {
        'n_estimators': nEstimators,
        'max_depth': maxDepth,
        'learning_rate': learningRate
    }
    
    allResults.append({
        'params': currentParams,
        'f1_macro': f1Val
    })
    
    if f1Val > bestF1Score:
        bestF1Score = f1Val
        bestParams = currentParams
        print(f"[{configIndex}/{totalConfigs}] ✨ Новый лучший F1-Macro: {f1Val:.5f}")
        print(f"  Параметры: {currentParams}")
    elif configIndex % 5 == 0:
        print(f"[{configIndex}/{totalConfigs}] Текущий F1: {f1Val:.5f}")

print(f"\n\nЛучшие параметры: {bestParams}")
print(f" Лучший F1-Macro на validation: {bestF1Score:.5f}")

## Оценка лучшей модели на validation

In [ ]:
bestLgbModel = xgb.XGBClassifier(
    **bestParams,
    **fixedParams
)

bestLgbModel.fit(xTrain, yTrain)

valPredictions = bestLgbModel.predict(xVal)
trainPredictions = bestLgbModel.predict(xTrain)

print("\n=== Лучшая LightGBM модель ===")
calculateMetrics(yTrain, trainPredictions, "Train")
bestF1Val = calculateMetrics(yVal, valPredictions, "Validation")

print("\nClassification Report (Validation):")
print(classification_report(yVal, valPredictions))

print("\nConfusion Matrix (Validation):")
print(confusion_matrix(yVal, valPredictions))

## Топ-10 конфигураций из Grid Search

In [ ]:
sortedResults = sorted(allResults, key=lambda x: x['f1_macro'], reverse=True)

print("\n=== Топ-10 конфигураций ===")
for i, result in enumerate(sortedResults[:10], 1):
    print(f"\n{i}. F1-Macro: {result['f1_macro']:.5f}")
    print(f"   Параметры: {result['params']}")

## Финальная модель на полном train

In [ ]:
print("\nОбучение финальной модели на полном train...\n")

finalModel = xgb.XGBClassifier(
    **bestParams,
    **fixedParams
    # n_estimators=1200,
    # max_depth=9,
    # learning_rate=0.02,
    # subsample=0.8,
    # colsample_bytree=0.8,
    # class_weight='balanced',
    # random_state=42,
    # n_jobs=-1,
    # verbose=-1
)

finalModel.fit(trainFeaturesSparse, yTarget)

finalTrainPred = finalModel.predict(trainFeaturesSparse).astype(int)
finalTestPred = finalModel.predict(testFeaturesSparse).astype(int)

print("\n=== Финальная модель на полном train ===")
calculateMetrics(yTarget, finalTrainPred, "Train")

print("\nClassification Report (Train):")
print(classification_report(yTarget, finalTrainPred))

print("\nConfusion Matrix (Train):")
print(confusion_matrix(yTarget, finalTrainPred))

print(f"\nПредсказано {len(finalTestPred)} меток для теста")
print(f"Распределение предсказанных классов: {np.bincount(finalTestPred)}")

## Сравнение с ручными разметками

In [ ]:
adaLabels = pd.read_csv('data/test_labeled_ada.csv')
gptLabels = pd.read_csv('data/test_labeled_gpt.csv')
grokLabels = pd.read_csv('data/test_labeled_grok.csv')

print("\n=== Сравнение с Ada (моя ручная разметка) ===")
print(f"Распределение классов Ada: {np.bincount(adaLabels['label'].astype(int))}")
calculateMetrics(adaLabels['label'].values, finalTestPred, "LGB vs Ada")
print("\nClassification Report:")
print(classification_report(adaLabels['label'].values, finalTestPred))
print("\nConfusion Matrix:")
print(confusion_matrix(adaLabels['label'].values, finalTestPred))

print("\n=== Сравнение с GPT ===")
print(f"Распределение классов GPT: {np.bincount(gptLabels['label'].astype(int))}")
calculateMetrics(gptLabels['label'].values, finalTestPred, "LGB vs GPT")
print("\nClassification Report:")
print(classification_report(gptLabels['label'].values, finalTestPred))
print("\nConfusion Matrix:")
print(confusion_matrix(gptLabels['label'].values, finalTestPred))

print("\n=== Сравнение с Grok ===")
print(f"Распределение классов Grok: {np.bincount(grokLabels['label'].astype(int))}")
calculateMetrics(grokLabels['label'].values, finalTestPred, "LGB vs Grok")
print("\nClassification Report:")
print(classification_report(grokLabels['label'].values, finalTestPred))
print("\nConfusion Matrix:")
print(confusion_matrix(grokLabels['label'].values, finalTestPred))

## Создание submission

In [ ]:
submissionDataFrame = pd.DataFrame({
    'id': testDataFrame['id'],
    'label': finalTestPred
})

submissionDataFrame.to_csv('submission_xgb_tuned.csv', index=False)
print("\nСохранен файл submission_xgb_tuned.csv")

print("\nПервые 10 предсказаний:")
print(submissionDataFrame.head(10))

print("\nРаспределение предсказанных классов:")
print(submissionDataFrame['label'].value_counts().sort_index())